# Classical Baselines
**TER — Quantum Computing for Machine Learning**

This notebook evaluates classical ML models as reference baselines for comparison with quantum classifiers (QCNN and Data Re-uploading).

**Models:**
- SVM (RBF kernel, grid-searched C and gamma)
- Random Forest (100 trees)
- NN-Small (16→8, ~36 params — matches Data Re-uploading)
- NN-Medium (12→6, ~51 params — matches QCNN)

**Two evaluation modes:**
- Standard: 3-class classification (normal / dos / injection)
- Zero-day: train on normal+dos, test on unseen injection

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import json
from pathlib import Path
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from classical.classical_baselines import (
    load_standard, load_zeroday,
    train_svm, train_random_forest, train_nn
)

DATA    = Path('../data')
RESULTS = Path('../results')
FIGS    = RESULTS / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)

matplotlib.rcParams.update({'figure.dpi': 130, 'font.size': 11})
print('Setup complete.')

## 1. Load Data

In [ ]:
# Standard split (3-class, quantum subsample: 500 train / 200 test)
X_tr_std, y_tr_std, X_te_std, y_te_std, le = load_standard()

# Zero-day split (binary train: normal+dos | test: injection unseen)
X_tr_zd, y_tr_zd, X_te_zd, y_te_zd = load_zeroday()

## 2. Standard Evaluation — 3-Class Classification

In [ ]:
std_results = []
trained_models = {}

# SVM
metrics, svm_model = train_svm(X_tr_std, y_tr_std, X_te_std, y_te_std, name='SVM')
std_results.append(metrics)
trained_models['SVM'] = svm_model

In [ ]:
# Random Forest
metrics, rf_model = train_random_forest(X_tr_std, y_tr_std, X_te_std, y_te_std)
std_results.append(metrics)
trained_models['RandomForest'] = rf_model

In [ ]:
# NN-Small (~36 params — matches Data Re-uploading parameter count)
metrics, nn_small = train_nn(
    X_tr_std, y_tr_std, X_te_std, y_te_std,
    hidden_layers=(16, 8),
    name='NN-Small (~36 params)'
)
std_results.append(metrics)
trained_models['NN-Small'] = nn_small

In [ ]:
# NN-Medium (~51 params — matches QCNN parameter count)
metrics, nn_med = train_nn(
    X_tr_std, y_tr_std, X_te_std, y_te_std,
    hidden_layers=(12, 6),
    name='NN-Medium (~51 params)'
)
std_results.append(metrics)
trained_models['NN-Medium'] = nn_med

## 3. Zero-Day Evaluation — Unseen Attack Class

In [ ]:
zd_results = []

metrics, _ = train_svm(X_tr_zd, y_tr_zd, X_te_zd, y_te_zd, name='SVM (zero-day)')
zd_results.append(metrics)

metrics, _ = train_random_forest(X_tr_zd, y_tr_zd, X_te_zd, y_te_zd, name='Random Forest (zero-day)')
zd_results.append(metrics)

metrics, _ = train_nn(X_tr_zd, y_tr_zd, X_te_zd, y_te_zd, hidden_layers=(16, 8), name='NN-Small (zero-day)')
zd_results.append(metrics)

metrics, _ = train_nn(X_tr_zd, y_tr_zd, X_te_zd, y_te_zd, hidden_layers=(12, 6), name='NN-Medium (zero-day)')
zd_results.append(metrics)

## 4. Results Summary

In [ ]:
df_std = pd.DataFrame(std_results)[['model', 'accuracy', 'f1']]
df_zd  = pd.DataFrame(zd_results)[['model', 'accuracy', 'f1']]

print('Standard (3-class):')
display(df_std.style.format({'accuracy': '{:.4f}', 'f1': '{:.4f}'})
              .background_gradient(subset=['accuracy', 'f1'], cmap='Greens'))

print('\nZero-day (injection unseen):')
display(df_zd.style.format({'accuracy': '{:.4f}', 'f1': '{:.4f}'})
              .background_gradient(subset=['accuracy', 'f1'], cmap='Oranges'))

## 5. Visualisation — Accuracy Comparison

In [ ]:
labels  = ['SVM', 'Random\nForest', 'NN-Small\n(~36p)', 'NN-Medium\n(~51p)']
acc_std = [r['accuracy'] for r in std_results]
acc_zd  = [r['accuracy'] for r in zd_results]

x   = np.arange(len(labels))
w   = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - w/2, acc_std, w, label='Standard (3-class)', color='#3498db', alpha=0.85)
bars2 = ax.bar(x + w/2, acc_zd,  w, label='Zero-day (unseen injection)', color='#e67e22', alpha=0.85)

for bar in bars1 + bars2:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f'{bar.get_height():.3f}',
        ha='center', va='bottom', fontsize=9
    )

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Accuracy')
ax.set_title('Classical Baselines — Standard vs Zero-Day')
ax.set_ylim(0, 1.1)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGS / 'classical_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {FIGS}/classical_comparison.png')

## 6. Confusion Matrix — Best Model (Standard)

In [ ]:
best_name = max(std_results, key=lambda r: r['accuracy'])['model']
best_key  = best_name.split()[0]   # 'SVM', 'Random', 'NN-Small', 'NN-Medium'
if best_key == 'Random': best_key = 'RandomForest'

best_model = trained_models[best_key]
y_pred     = best_model.predict(X_te_std)
cm         = confusion_matrix(y_te_std, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Confusion Matrix — {best_name}')
plt.tight_layout()
plt.savefig(FIGS / 'classical_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Save Results

In [ ]:
logs_dir = RESULTS / 'logs'
logs_dir.mkdir(parents=True, exist_ok=True)

all_results = {'standard': std_results, 'zeroday': zd_results}
with open(logs_dir / 'classical_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print('Results saved → results/logs/classical_results.json')
print('These will be loaded again in the final comparison notebook (quantum vs classical).')